# 🤖 ESG Classification Modeling
**Objective:** Predict ESG performance category (Excellent, Good, Average, Poor) from environmental metrics.

**Features Used:** CarbonEmissions, WaterUsage, EnergyConsumption

**Models:** Logistic Regression, Random Forest, Gradient Boosting, SVM, KNN

---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
 accuracy_score, precision_score, recall_score, f1_score,
 classification_report, confusion_matrix, roc_auc_score,
 ConfusionMatrixDisplay
)

import joblib
import os

print('Libraries loaded successfully!')

## 2. Load & Inspect Data

In [ ]:
# Load processed dataset (already has ESG_Category column from EDA notebook)
# Try processed first, fallback to raw
try:
 df = pd.read_csv('../esg_processed.csv')
 print('Loaded esg_processed.csv')
except FileNotFoundError:
 df = pd.read_csv('../raw_data.csv')
 print('Loaded raw_data.csv')

print(f'Shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
df.head()

In [ ]:
print('=== Data Info ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())

## 3. Preprocessing & Feature Engineering

In [ ]:
# Create ESG_Category if not already present
if 'ESG_Category' not in df.columns:
 def categorize_esg(score):
 if score > 75: return 'Excellent'
 elif score >= 50: return 'Good'
 elif score >= 25: return 'Average'
 else: return 'Poor'
 df['ESG_Category'] = df['ESG_Overall'].apply(categorize_esg)
 print('ESG_Category column created.')
else:
 print('ESG_Category column already exists.')

print('\nClass Distribution:')
print(df['ESG_Category'].value_counts())

In [ ]:
fig = px.histogram(
 df, x='ESG_Category',
 color='ESG_Category',
 category_orders={'ESG_Category': ['Excellent', 'Good', 'Average', 'Poor']},
 title='ESG Category Distribution',
 color_discrete_map={
 'Excellent': '#2ecc71', 'Good': '#3498db',
 'Average': '#f39c12', 'Poor': '#e74c3c'
 }
)
fig.show()

In [ ]:
# Define features and target
FEATURES = ['CarbonEmissions', 'WaterUsage', 'EnergyConsumption']
TARGET = 'ESG_Category'

# Drop rows with missing values in selected columns
df_model = df[FEATURES + [TARGET]].dropna()
print(f'Modeling dataset shape: {df_model.shape}')

X = df_model[FEATURES]
y = df_model[TARGET]

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f'Classes: {le.classes_}')

In [ ]:
# Visualize feature distributions by ESG category
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, feature in enumerate(FEATURES):
 for category in ['Excellent', 'Good', 'Average', 'Poor']:
 subset = df_model[df_model[TARGET] == category][feature]
 axes[i].hist(subset, alpha=0.5, label=category, bins=30)
 axes[i].set_title(f'{feature} by ESG Category')
 axes[i].set_xlabel(feature)
 axes[i].legend()
plt.tight_layout()
plt.savefig('../outputs/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/feature_distributions.png')

## 4. Train-Test Split & Feature Scaling

In [ ]:
# Stratified split to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
 X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {X_train_scaled.shape}')
print(f'Test set: {X_test_scaled.shape}')

## 5. Model Training & Comparison

In [ ]:
# Define all models
models = {
 'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
 'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
 'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
 'SVM': SVC(kernel='rbf', probability=True, random_state=42),
 'KNN': KNeighborsClassifier(n_neighbors=5)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
 model.fit(X_train_scaled, y_train)
 y_pred = model.predict(X_test_scaled)
 cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
 results.append({
 'Model': name,
 'Test Accuracy': accuracy_score(y_test, y_pred),
 'F1 Score (Weighted)': f1_score(y_test, y_pred, average='weighted'),
 'CV Mean': cv_scores.mean(),
 'CV Std': cv_scores.std()
 })
 print(f'{name}: Acc={accuracy_score(y_test, y_pred):.4f} | CV={cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
print('\n=== Model Comparison ===')
results_df

In [ ]:
fig = px.bar(
 results_df.melt(id_vars='Model', value_vars=['Test Accuracy', 'F1 Score (Weighted)', 'CV Mean']),
 x='Model', y='value', color='variable', barmode='group',
 title='Model Performance Comparison',
 labels={'value': 'Score', 'variable': 'Metric'},
 color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(yaxis_range=[0, 1])
fig.show()

## 6. Hyperparameter Tuning (Best Model)

In [ ]:
# Tune Random Forest (typically best performer)
param_grid = {
 'n_estimators': [100, 200, 300],
 'max_depth': [None, 10, 20, 30],
 'min_samples_split': [2, 5, 10],
 'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
 rf, param_grid, cv=5, scoring='accuracy',
 n_jobs=-1, verbose=1
)
grid_search.fit(X_train_scaled, y_train)

print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV Accuracy: {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_

## 7. Final Model Evaluation

In [ ]:
y_pred_best = best_model.predict(X_test_scaled)
y_proba_best = best_model.predict_proba(X_test_scaled)

print('=== Final Model (Tuned Random Forest) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_best):.4f}')
print(f'F1 (weighted): {f1_score(y_test, y_pred_best, average="weighted"):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_best, average="weighted"):.4f}')
print(f'Recall : {recall_score(y_test, y_pred_best, average="weighted"):.4f}')
print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix - Tuned Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/confusion_matrix.png')

In [ ]:
# Feature Importance
importances = best_model.feature_importances_
feat_imp_df = pd.DataFrame({
 'Feature': FEATURES,
 'Importance': importances
}).sort_values('Importance', ascending=False)

fig = px.bar(
 feat_imp_df, x='Feature', y='Importance',
 color='Importance', color_continuous_scale='Greens',
 title='Feature Importance - Random Forest',
 text='Importance'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.show()

## 8. Save Models & Artifacts

In [ ]:
os.makedirs('../models', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

# Save model artifacts
joblib.dump(best_model, '../models/best_classification_model.pkl')
joblib.dump(scaler, '../models/classification_scaler.pkl')
joblib.dump(le, '../models/label_encoder.pkl')

# Save results CSV
results_df.to_csv('../outputs/classification_results.csv', index=False)

print('Models saved:')
print(' - models/best_classification_model.pkl')
print(' - models/classification_scaler.pkl')
print(' - models/label_encoder.pkl')
print('Results saved: outputs/classification_results.csv')

## 9. Key Findings

In [ ]:
print('=== KEY FINDINGS ===')
print(f'Best Model: Tuned Random Forest')
print(f'Best Test Accuracy: {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Best F1 Score: {f1_score(y_test, y_pred_best, average="weighted"):.4f}')
print(f'\nMost Important Feature: {feat_imp_df.iloc[0]["Feature"]} ({feat_imp_df.iloc[0]["Importance"]:.3f})')
print(f'\nClass distribution in test set:')
unique, counts = np.unique(y_test, return_counts=True)
for cls, cnt in zip(le.classes_, counts):
 print(f' {cls}: {cnt} samples')